In [1]:
import pandas as pd
import json

def check_text_distribution(jsonl_file):
    word_counts = []
    
    print(f"Reading data from {jsonl_file}...\n")
    
    # Read the file line by line to manage memory efficiently
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                text = record.get("text", "")
                
                # A simple approximation for word count is splitting by whitespace
                count = len(text.split())
                word_counts.append(count)
            except json.JSONDecodeError:
                continue

    # Convert the list to a pandas Series for easy statistical calculations
    df_counts = pd.Series(word_counts)
    
    if df_counts.empty:
        print("No valid records found to analyze.")
        return

    # Calculate basic statistics
    total_speeches = len(df_counts)
    min_words = df_counts.min()
    max_words = df_counts.max()
    mean_words = df_counts.mean()
    median_words = df_counts.median()
    
    # Calculate how many speeches fit your < 250 words criteria
    under_250 = (df_counts < 250).sum()
    percentage_under_250 = (under_250 / total_speeches) * 100

    # Print the distribution summary
    print("--- Word Count Distribution Summary ---")
    print(f"Total Speeches Processed: {total_speeches:,}")
    print(f"Minimum Word Count:       {min_words:,} words")
    print(f"Maximum Word Count:       {max_words:,} words")
    print(f"Average (Mean) Length:    {mean_words:.1f} words")
    print(f"Median Length:            {median_words:.1f} words")
    
    print("\n--- Percentiles ---")
    print(f"25% of speeches are shorter than: {df_counts.quantile(0.25):.0f} words")
    print(f"50% of speeches are shorter than: {df_counts.quantile(0.50):.0f} words (Median)")
    print(f"75% of speeches are shorter than: {df_counts.quantile(0.75):.0f} words")
    print(f"90% of speeches are shorter than: {df_counts.quantile(0.90):.0f} words")
    print(f"99% of speeches are shorter than: {df_counts.quantile(0.99):.0f} words")
    
    print("\n--- Your Specific Task ---")
    print(f"Speeches under 250 words: {under_250:,} out of {total_speeches:,} ({percentage_under_250:.2f}%)")

# --- Execution ---
# Point this to the file generated by the previous script
FILE_PATH = 'speeches_with_party.jsonl'

check_text_distribution(FILE_PATH)

Reading data from speeches_with_party.jsonl...

--- Word Count Distribution Summary ---
Total Speeches Processed: 12,976,651
Minimum Word Count:       1 words
Maximum Word Count:       73,140 words
Average (Mean) Length:    118.2 words
Median Length:            23.0 words

--- Percentiles ---
25% of speeches are shorter than: 9 words
50% of speeches are shorter than: 23 words (Median)
75% of speeches are shorter than: 80 words
90% of speeches are shorter than: 298 words
99% of speeches are shorter than: 1446 words

--- Your Specific Task ---
Speeches under 250 words: 11,463,451 out of 12,976,651 (88.34%)


In [2]:
import json
from collections import Counter

def check_label_distribution(jsonl_file):
    # Counter will automatically keep track of how many times it sees each label
    party_counts = Counter()
    
    print(f"Reading data from {jsonl_file}...\n")
    
    # Read line by line to keep memory usage low
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                # Grab the label, default to 'Unknown' if missing
                label = record.get("label", "Unknown")
                
                # Clean up whitespace just in case
                label = label.strip() if isinstance(label, str) else str(label)
                
                party_counts[label] += 1
            except json.JSONDecodeError:
                continue

    total_speeches = sum(party_counts.values())
    
    if total_speeches == 0:
        print("No valid records found.")
        return

    # Print the results
    print("--- Party Label Distribution ---")
    print(f"Total Speeches Processed: {total_speeches:,}\n")
    
    # .most_common() sorts the results from highest count to lowest
    for party, count in party_counts.most_common():
        percentage = (count / total_speeches) * 100
        
        # Make the output easier to read for the main parties
        party_name = party
        if party == 'R':
            party_name = 'Republican (R)'
        elif party == 'D':
            party_name = 'Democrat (D)'
            
        print(f"{party_name}: {count:,} speeches ({percentage:.2f}%)")

# --- Execution ---
# Point this to the file generated by your merging script
FILE_PATH = 'speeches_with_party.jsonl'

check_label_distribution(FILE_PATH)


Reading data from speeches_with_party.jsonl...

--- Party Label Distribution ---
Total Speeches Processed: 12,976,651

Democrat (D): 6,824,644 speeches (52.59%)
Republican (R): 6,022,962 speeches (46.41%)
P: 44,703 speeches (0.34%)
I: 39,820 speeches (0.31%)
S: 15,923 speeches (0.12%)
F: 10,076 speeches (0.08%)
N: 6,437 speeches (0.05%)
A: 5,961 speeches (0.05%)
C: 3,474 speeches (0.03%)
L: 1,744 speeches (0.01%)
U: 907 speeches (0.01%)
